# Managing Versioned Files with VersionedClass

## Introduction

This tutorial demonstrates how to use `VersionedClass` to build a system that can open and process files with different versions (e.g., JSON configuration files).

We will build a `FileManager` that inspects a file's content to determine its version and dispatches the handling to the appropriate subclass.


## Importing Modules


In [1]:
import json
import tempfile
from pathlib import Path
from typing import Any
from classversioning import TriNumberVersion, VersionedClass, VersionRegistry


## Defining the Head Class

The head class `FileManager` defines the version type and the `get_version_from_object` method. This method inspects a file path, reads the content, and extracts the version.


In [2]:
class FileManager(VersionedClass):
    """Head class for file managers.

    This class defines how to extract the version from a file.
    """
    class_registration = True
    class_registry_type = VersionRegistry
    VERSION_TYPE = TriNumberVersion

    @classmethod
    def get_version_from_object(cls, obj: Path | str) -> TriNumberVersion | None:
        """Extracts version from the file.

        Args:
            obj: The file path.

        Returns:
            The version object found in the file, or None if not found.
        """
        path = Path(obj)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")

        try:
            with path.open("r") as f:
                data = json.load(f)
                version_str = data.get("version", "0.0.0")
                parts = map(int, version_str.split('.'))
                return TriNumberVersion(*parts)
        except Exception as e:
            print(f"Error reading version from file: {e}")
            return None

    def __init__(self, obj: Path | str, **kwargs: Any) -> None:
        self.file_path = Path(obj)

    def load_data(self) -> Any:
        """Abstract method to load data."""
        raise NotImplementedError


## Defining Versioned Subclasses

Now we define specific managers for Version 1.0.0 and Version 2.0.0. Each subclass implements `load_data` differently.


In [3]:
class FileManagerV1(FileManager):
    """File manager for version 1.0.0 files."""
    VERSION = TriNumberVersion(1, 0, 0)

    def load_data(self) -> dict[str, Any]:
        with self.file_path.open("r") as f:
            data = json.load(f)
        print(f"V1 Manager loading data: {data}")
        return data


class FileManagerV2(FileManager):
    """File manager for version 2.0.0 files."""
    VERSION = TriNumberVersion(2, 0, 0)

    def load_data(self) -> dict[str, Any]:
        with self.file_path.open("r") as f:
            data = json.load(f)
        # V2 might need some transformation
        data["processed"] = True
        print(f"V2 Manager loading data with processing: {data}")
        return data


## Basic Usage

We create temporary files with different versions and demonstrate how `FileManager` automatically dispatches to the correct subclass.


In [4]:
with tempfile.TemporaryDirectory() as tmp_dir:
    path_v1 = Path(tmp_dir) / "data_v1.json"
    path_v2 = Path(tmp_dir) / "data_v2.json"

    with path_v1.open("w") as f:
        json.dump({"version": "1.0.0", "content": "old data"}, f)

    with path_v2.open("w") as f:
        json.dump({"version": "2.0.0", "content": "new data"}, f)

    # Dispatching
    manager1 = FileManager(obj=path_v1)
    print(f"Manager for {path_v1.name} is instance of: {manager1.__class__.__name__}")
    manager1.load_data()

    manager2 = FileManager(obj=path_v2)
    print(f"Manager for {path_v2.name} is instance of: {manager2.__class__.__name__}")
    manager2.load_data()


Manager for data_v1.json is instance of: FileManagerV1
V1 Manager loading data: {'version': '1.0.0', 'content': 'old data'}
Manager for data_v2.json is instance of: FileManagerV2
V2 Manager loading data with processing: {'version': '2.0.0', 'content': 'new data', 'processed': True}


## Handling Edge Cases

We can also see how the system behaves when encountering unknown or future versions.


In [5]:
with tempfile.TemporaryDirectory() as tmp_dir:
    path_future = Path(tmp_dir) / "data_future.json"
    with path_future.open("w") as f:
        json.dump({"version": "5.0.0", "content": "future data"}, f)

    print(f"Attempting to load {path_future.name} (Version 5.0.0)...")
    try:
        manager = FileManager(obj=path_future)
        print(f"-> Dispatched to: {manager.__class__.__name__} (Version {manager.VERSION})")
        print("   (Note: Default behavior is to find the latest version <= requested version)")
    except Exception as e:
        print(f"-> Caught error: {e}")

    path_old = Path(tmp_dir) / "data_old.json"
    with path_old.open("w") as f:
        json.dump({"version": "0.1.0", "content": "ancient data"}, f)

    print(f"\nAttempting to load {path_old.name} (Version 0.1.0)...")
    try:
        manager = FileManager(obj=path_old)
        print(f"-> Dispatched to: {manager.__class__.__name__} (Base Class)")
        # The base class does not implement load_data
        manager.load_data()
    except Exception as e:
        print(f"-> Caught expected error when calling load_data: {e}")


Attempting to load data_future.json (Version 5.0.0)...
-> Dispatched to: FileManagerV2 (Version 2.0.0)
   (Note: Default behavior is to find the latest version <= requested version)

Attempting to load data_old.json (Version 0.1.0)...
-> Dispatched to: FileManager (Base Class)
-> Caught expected error when calling load_data: 


## Conclusion

This tutorial showed how to extend `VersionedClass` to manage file-based versioning. By implementing `get_version_from_object`, you can create powerful dispatch mechanisms that abstract away version differences.
